# Laboratorio 10 — Dos escritores al mismo tiempo

**Duración: 45 minutos.**

Hasta ahora cada tabla tuvo un solo escritor, tu cuaderno. Hoy vas a poner **dos
escritores** sobre la misma tabla, a la vez, y a propósito.

No es un ejercicio de laboratorio: es lo que pasa todas las noches en una plataforma de
datos. Dos procesos de carga que arrancan a la misma hora y tocan la misma tabla. La
pregunta del día es qué ocurre entonces, y la respuesta tiene dos partes que conviene no
confundir.

---

### En el almacén

Dos personas quieren anotar en la misma libreta al mismo tiempo.

- Si cada una anota **una cosa nueva** al final, no hay problema: las dos anotaciones
  caben y quedan las dos.
- Si las dos quieren **corregir el mismo renglón**, hay problema: alguna de las dos va a
  escribir encima de la otra, y eso no puede pasar sin que nadie se entere.

Iceberg trata esos dos casos de forma distinta, y hoy vas a ver los dos.

La tabla de hoy se llama `curso.escritores_lab10`. No está en tu espacio sino en `curso`,
el espacio de las tablas de referencia. Si quieres repetir el laboratorio desde el
principio, déjala como estaba con `bin/reiniciar-lab.sh 10`.


## Paso 0 — La tabla compartida

Primero mira lo que hay. No hace falta `USE`: la tabla se nombra completa, con su
database adelante, porque no es tuya.

In [ ]:
%%sql
-- Celda 0.1



Cuatro contribuyentes, los mismos que viste en otros laboratorios. Y una sola página
en la libreta, la de la carga inicial.

In [ ]:
%%sql
-- Celda 0.2



## Paso 1 — Dos escritores insertan a la vez

Ahora el primer experimento. Para que haya **dos escritores de verdad** hace falta que
dos cosas escriban al mismo tiempo, y tú eres una sola persona. Esta celda resuelve eso:
lanza **dos hilos**, que son dos líneas de trabajo que corren en paralelo dentro de tu
cuaderno, y cada uno inserta su fila.

Ya viene escrita. Léela antes de correrla: `threading.Thread` crea cada hilo, `start()`
los echa a andar y `join()` espera a que los dos terminen.

Los dos van con el **mismo RUT**, y es a propósito: la tabla va a quedar con dos filas del
mismo contribuyente. Hoy eso no importa, porque lo que estamos mirando es qué pasa cuando
dos escriben a la vez, no la llave del padrón.

> **También se puede hacer a mano, con dos pestañas.** Para que sean dos escritores de
> verdad, cada pestaña necesita su propio kernel, y el mismo cuaderno abierto dos veces
> comparte uno solo: las dos celdas se ejecutarían una detrás de otra en el mismo Spark.
> Por eso la segunda pestaña abre **una copia**. En el explorador de archivos, clic derecho
> sobre `lab-10.ipynb`, **Duplicate**, y abre `lab-10-Copy1.ipynb` en otra pestaña del
> navegador: la copia arranca su propio Spark. En una celda nueva de cada pestaña escribe
> el mismo `INSERT` con un nombre distinto y ejecútalas una tras otra, lo más rápido que
> puedas. Los hilos son la versión que no depende de tus dedos.

In [ ]:
# Celda 1.1
import threading

def escribe(nombre):
    spark.sql(
        f"INSERT INTO curso.escritores_lab10 VALUES "
        f"('99000001-1', 'Escritor {nombre} SpA', 'MICRO')"
    )
    print(nombre, "escribio")

hilos = [threading.Thread(target=escribe, args=(n,)) for n in ("uno", "dos")]
for h in hilos:
    h.start()
for h in hilos:
    h.join()

print("los dos terminaron")

A ninguno de los dos le falló. Cuenta las filas.

In [ ]:
%%sql
-- Celda 1.2



Están las seis: las cuatro iniciales más una por cada hilo. Ahora mira la libreta.

In [ ]:
%%sql
-- Celda 1.3



## Paso 2 — Los dos corrigen la misma fila

Una página nueva por cada hilo, las dos `append`, en el orden en que llegaron.
**Ninguno perdió su fila y ninguno tuvo que esperar al otro.**

Ahora el segundo experimento, y es el interesante. Esta vez los dos no agregan algo
nuevo: **corrigen la misma fila**, al mismo tiempo, y cada uno le pone un segmento
distinto.

In [ ]:
# Celda 2.1
import threading

def corrige(nombre, segmento):
    try:
        spark.sql(
            f"UPDATE curso.escritores_lab10 SET segmento = '{segmento}' "
            f"WHERE rut = '77746521-K'"
        )
        print(nombre, "corrigio a", segmento)
    except Exception as error:
        lineas = str(error).splitlines()
        print(nombre, "NO pudo:", lineas[0][:90])
        causa = [l.strip() for l in lineas if "ValidationException" in l]
        if causa:
            print(causa[0])

hilos = [
    threading.Thread(target=corrige, args=("uno", "GRANDE")),
    threading.Thread(target=corrige, args=("dos", "PEQUENA")),
]
for h in hilos:
    h.start()
for h in hilos:
    h.join()

print("los dos terminaron")

**A uno de los dos le falló.** Mira lo que imprimió el que perdió: primero el mensaje
corto, `NO pudo: An error occurred while calling ...`, y debajo la línea de la causa, la que
empieza con `Caused by` y dice `ValidationException: Found conflicting files`. Las dos
líneas con `ERROR ReplaceDataExec` que salen antes son Spark avisando que descarta la
escritura del que perdió. Cuál de los dos gana no se sabe de antemano: depende de cuál
llega primero.

No es un error de la celda. Es Iceberg diciendo que el otro hilo cambió esa misma fila
mientras este preparaba su cambio, y que por eso **no va a escribir encima**.

Mira cómo quedó.

In [ ]:
%%sql
-- Celda 2.2



Uno solo ganó, y el segmento que quedó es el suyo. Ahora la historia completa.

In [ ]:
%%sql
-- Celda 2.3



## Paso 3 — Concurrencia optimista

Lo que acabas de ver tiene nombre: **concurrencia optimista**.

**Iceberg no bloquea.** No existe un candado que un proceso ponga sobre la tabla para que
los demás esperen. Cada escritor prepara su cambio por su cuenta, sin pedir permiso, y
solo al final intenta anotarlo en la libreta.

Ahí pasa una de tres cosas:

- **Nadie tocó nada mientras tanto.** Se anota y listo.
- **Alguien anotó algo que no se pisa con lo tuyo**, como en el paso 1, donde cada hilo
  agregó una fila distinta. Iceberg **reintenta solo**, contra la página nueva, y lo
  consigue. Por eso el paso 1 no falló y nadie vio nada.
- **Alguien tocó los mismos archivos que tú**, como en el paso 2. Ahí no hay reintento que
  valga: reintentar sería escribir encima de una corrección ajena. Iceberg **falla a
  propósito**.

Esa tercera es la que importa en producción, y es la que casi nadie espera la primera vez.

---

### Qué hace un proceso de carga bien escrito

Si el commit le rebota:

1. **No se rinde ni sigue como si nada.** Un error de commit no es un aviso.
2. **Vuelve a leer la tabla** y rehace su cálculo sobre lo que hay ahora, no sobre lo que
   había cuando empezó.
3. **Reintenta unas pocas veces**, con una espera que crece entre intento e intento.
4. **Si sigue rebotando, avisa.** Que dos procesos peleen siempre por las mismas filas es
   un problema de diseño, no de suerte, y hay que arreglarlo arriba: separando las cargas
   por partición, por ejemplo, para que no se toquen.

Lo que **no** hay que hacer es el reintento ciego sin releer. Eso reintroduce justo lo que
Iceberg estaba evitando.

## Cierre

Dos cosas distintas, y conviene no confundirlas nunca más:

| Lo que pasó | Qué hizo Iceberg | Qué viste |
|---|---|---|
| Cada hilo agregó una fila distinta | reintenta solo, si hace falta | nada, funcionó |
| Los dos corrigieron la misma fila | falló a propósito | `ValidationException` |

**Ninguna de las dos perdió un dato.** Ni en la primera, donde están las seis filas, ni
en la segunda, donde el que ganó dejó su corrección y el otro se enteró de que no pudo.

Eso es lo que hace que una tabla Iceberg aguante que varios procesos escriban sin
coordinarse. No hay un coordinador: hay una regla, y la regla es que nadie escribe encima
de lo que no leyó.
